:orphan:
# Plotting Clustering
Figures for the clustering section of the Unsupervised Learning chapter.

In [ ]:
from myst_nb import glue
import numpy as np
import matplotlib.pyplot as plt


## Style

In [ ]:
# ---------------------------------------------------------------- style
STYLE = {
    "font.family": ["Latin Modern Roman", "serif"],  # falls back if LM not installed
    "mathtext.fontset": "cm",
}
plt.rcParams.update(STYLE)
INK, AX, DASH = "#1f2a3a", "#374151", "#b4bcc8"
GREY_STEPS = "#6b7280"            # grey of the k-means walkthrough points
GREY_INTRO = "#8a8a8a"            # neutral grey of the intro figure
RED_P, GREEN_P = "#ec7b70", "#62bd86"   # point colours (walkthrough)
RED_C, GREEN_C = "#c8372d", "#1f8a4c"   # centroid colours (walkthrough)
INTRO_COLS = ["#e2655a", "#4fae78", "#5b8fd6", "#e3a53c"]  # red, green, blue, amber
L = 16                            # axes run from 0 to L


def _box(ax, ticks=True):
    ax.set_xlim(0, L); ax.set_ylim(0, L); ax.set_aspect("equal")
    if ticks:
        ax.set_xticks(range(0, L + 1, 2)); ax.set_yticks(range(0, L + 1, 2))
        ax.tick_params(colors=AX, labelsize=12, length=4, width=1)
        ax.set_xlabel("$x_1$", fontsize=15, color=INK)
        ax.set_ylabel("$x_2$", fontsize=15, color=INK, rotation=0, labelpad=12)
    else:
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel("$x_1$", fontsize=16, color=INK, labelpad=8)
        ax.set_ylabel("$x_2$", fontsize=16, color=INK, rotation=0, labelpad=14)
    for s in ax.spines.values():
        s.set_color(AX); s.set_linewidth(1.2)


## Introduction: unlabelled data and clusters

In [ ]:
# ======================================================= intro: grey -> 4 clusters
def intro_data(seed=5):
    """620 points in 4 clusters of different shapes. Returns X, true labels."""
    rng = np.random.default_rng(seed)
    spec = [  # centre, sx, sy, angle (deg), n
        ((4.4, 10.2), 1.85, 0.50, 35, 170),   # long diagonal streak  (red)
        ((10.9, 12.2), 0.55, 0.50, 0, 110),   # small dense blob      (green)
        ((11.3, 5.3), 1.35, 1.20, 0, 200),    # big diffuse cloud     (blue)
        ((4.8, 4.4), 1.05, 0.50, -18, 140),   # flat tilted ellipse   (amber)
    ]
    X, lab = [], []
    for k, (c, sx, sy, ang, m) in enumerate(spec):
        a = np.deg2rad(ang)
        R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
        X.append(rng.normal(0, 1, (m, 2)) * [sx, sy] @ R.T + c); lab += [k] * m
    X, lab = np.vstack(X), np.array(lab)
    keep = (X > 0.4).all(1) & (X < L - 0.4).all(1)
    return X[keep], lab[keep]


def kmeans(X, K, n_init=20, max_iter=100, seed=0):
    """Plain k-means following the pseudocode, with n_init random restarts.

    Returns the labels c (length m) and centroids mu (K x n) of the run
    with the lowest within-cluster sum of squared distances J.
    """
    rng = np.random.default_rng(seed)
    best_J = np.inf
    for _ in range(n_init):
        # Initialisation: K distinct data points chosen at random
        mu = X[rng.choice(len(X), K, replace=False)].astype(float)
        c = None
        for _ in range(max_iter):
            # E-step: assign each point to its nearest centroid
            d2 = ((X[:, None, :] - mu[None, :, :]) ** 2).sum(-1)
            c_new = d2.argmin(1)
            if c is not None and (c_new == c).all():
                break                                   # no assignment changed
            c = c_new
            # M-step: move each centroid to the mean of its cluster
            mu = np.array([X[c == k].mean(0) if (c == k).any() else mu[k]
                           for k in range(K)])
        J = ((X - mu[c]) ** 2).sum()
        if J < best_J:
            best_J, best_c, best_mu = J, c, mu
    return best_c, best_mu


def intro_kmeans_labels(X, lab):
    """k-means (K=4) labels, mapped onto the true-label colours."""
    c, _ = kmeans(X, 4)
    m = {k: np.bincount(lab[c == k]).argmax() for k in range(4)}
    return np.array([m[k] for k in c])


def clustering_intro_fig(labels, X):
    """Before -> after: grey unlabelled data, arrow, coloured clusters."""
    fig = plt.figure(figsize=(11, 4.6), dpi=220)
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.55, 1], wspace=0)
    ax_l, ax_a, ax_r = (fig.add_subplot(gs[0, i]) for i in range(3))
    for ax, cols in ((ax_l, GREY_INTRO), (ax_r, [INTRO_COLS[k] for k in labels])):
        _box(ax, ticks=False)
        ax.scatter(X[:, 0], X[:, 1], c=cols, s=9, edgecolor="none", alpha=0.9)
    # block arrow, proportions copied from the reference arrow:
    # shaft thickness 0.16 L, head height 0.36 L, head length 0.31 L
    ax_a.axis("off")
    # equal scaling in x and y so the arrow keeps its proportions
    W, H = fig.get_size_inches(); bb = ax_a.get_position()
    r = (bb.height * H) / (bb.width * W)
    ax_a.set_xlim(0, 1); ax_a.set_ylim(-r / 2, r / 2)
    Lr, xc = 0.42, 0.41                    # arrow length, centre
    x0, x1 = xc - Lr / 2, xc + Lr / 2
    sh, hh, hl = 0.082 * Lr, 0.178 * Lr, 0.31 * Lr
    ax_a.add_patch(plt.Polygon(
        [(x0, -sh), (x1 - hl, -sh), (x1 - hl, -hh), (x1, 0),
         (x1 - hl, hh), (x1 - hl, sh), (x0, sh)],
        closed=True, fc="#b5b5b5", ec="none"))
    return fig


X_intro, lab_intro = intro_data()


### True groups (introduction)

In [ ]:
fig = clustering_intro_fig(lab_intro, X_intro)

glue("plot_clustering_intro", fig, display=False)


### k-means output (limitations)

In [ ]:
fig = clustering_intro_fig(intro_kmeans_labels(X_intro, lab_intro), X_intro)

glue("plot_clustering_intro_kmeans", fig, display=False)


## Cosine similarity: clusters by direction

In [ ]:
COS_COLS = ["#e2655a", "#4fae78", "#5b8fd6", "#e3a53c",
            "#9b72c9", "#3fb0ad", "#d97fae"]    # 7 clusters
COS_L = 8                                      # axes run from -COS_L to COS_L
FRAME = "#8b93a1"                              # lighter grey for the outer frame


def cosine_data(seed=7):
    """Points along 7 irregular directions (with one wide empty sector),
    at very different distances from the origin."""
    rng = np.random.default_rng(seed)
    #        angle(deg) spread(deg)  n   r_min  r_scale
    spec = [(   18,       3.5,      40,  1.2,   2.6),
            (   47,       7.5,      24,  1.0,   1.6),
            (  112,       2.8,      46,  2.0,   2.2),
            (  138,       5.5,      28,  0.9,   1.4),
            (  196,       4.5,      38,  1.5,   3.0),
            (  231,       9.0,      20,  1.0,   1.2),
            (  342,       4.0,      34,  1.8,   2.4)]
    X, lab, dirs = [], [], []
    for k, (a, sp, n, r0, rs) in enumerate(spec):
        th0 = np.deg2rad(a)
        P = np.empty((0, 2))
        while len(P) < n:                       # keep only points inside the frame
            th = th0 + np.deg2rad(sp) * rng.normal(0, 1, n)
            r = r0 + rng.gamma(1.6, rs, n)      # long tail: some points far out
            Q = np.c_[r * np.cos(th), r * np.sin(th)]
            P = np.vstack([P, Q[(np.abs(Q) < COS_L - 0.5).all(1)]])
        X.append(P[:n]); lab += [k] * n
        dirs.append(th0)
    return np.vstack(X), np.array(lab), np.array(dirs)


def _cos_box(ax):
    ax.set_xlim(-COS_L, COS_L); ax.set_ylim(-COS_L, COS_L); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(FRAME); s.set_linewidth(1.0)
    # central cross: the feature axes through the origin
    ax.axhline(0, color=AX, lw=1.0, zorder=1); ax.axvline(0, color=AX, lw=1.0, zorder=1)
    ax.text(COS_L - 0.35, 0.3, "$x_1$", fontsize=16, color=INK, ha="right", va="bottom")
    ax.text(0.35, COS_L - 0.35, "$x_2$", fontsize=16, color=INK, ha="left", va="top")


def cosine_clusters_fig():
    """Before -> after: grey points, arrow, points coloured by direction."""
    X, lab, dirs = cosine_data()
    fig = plt.figure(figsize=(11, 4.6), dpi=220)
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.55, 1], wspace=0)
    ax_l, ax_a, ax_r = (fig.add_subplot(gs[0, i]) for i in range(3))
    for ax in (ax_l, ax_r):
        _cos_box(ax)
    ax_l.scatter(X[:, 0], X[:, 1], c=GREY_INTRO, s=11, edgecolor="none", alpha=0.9, zorder=3)
    for k, th in enumerate(dirs):           # faint ray along each cluster direction
        ax_r.plot([0, 2 * COS_L * np.cos(th)], [0, 2 * COS_L * np.sin(th)],
                  color=COS_COLS[k], lw=0.9, alpha=0.45, zorder=2)
    ax_r.scatter(X[:, 0], X[:, 1], c=[COS_COLS[k] for k in lab], s=11,
                 edgecolor="none", alpha=0.9, zorder=3)
    # same arrow as the intro figure
    ax_a.axis("off")
    W, H = fig.get_size_inches(); bb = ax_a.get_position()
    r = (bb.height * H) / (bb.width * W)
    ax_a.set_xlim(0, 1); ax_a.set_ylim(-r / 2, r / 2)
    Lr, xc = 0.42, 0.50
    x0, x1 = xc - Lr / 2, xc + Lr / 2
    sh, hh, hl = 0.082 * Lr, 0.178 * Lr, 0.31 * Lr
    ax_a.add_patch(plt.Polygon(
        [(x0, -sh), (x1 - hl, -sh), (x1 - hl, -hh), (x1, 0),
         (x1 - hl, hh), (x1 - hl, sh), (x0, sh)],
        closed=True, fc="#b5b5b5", ec="none"))
    return fig


fig = cosine_clusters_fig()

glue("plot_cosine_clusters", fig, display=False)


## k-means step by step
Dataset A to H, $k = 2$, centroids initialised on A and B.

In [ ]:
# ======================================================= k-means walkthrough
POINTS = {"A": (4, 10), "B": (6, 11), "C": (2, 9), "D": (5, 7),
          "E": (8, 3), "F": (10, 6), "G": (12, 4), "H": (14, 8)}
_N = list(POINTS)
_X = np.array([POINTS[n] for n in _N], float)
_MU1, _MU2 = r"$\boldsymbol{\mu}_1$", r"$\boldsymbol{\mu}_2$"
# label offsets (dx, dy) relative to the centroid "+"
_INIT, _TL, _TR = (-0.55, 0.72), (-0.5, 0.5), (0.5, 0.5)


def _assign(C):
    return ((_X[:, None] - C[None]) ** 2).sum(-1).argmin(1)


def kmeans_trace():
    """Centroids and assignments of the walkthrough (init on A and B)."""
    C0 = np.array([POINTS["A"], POINTS["B"]], float)
    a1 = _assign(C0); C1 = np.array([_X[a1 == k].mean(0) for k in (0, 1)])
    a2 = _assign(C1); C2 = np.array([_X[a2 == k].mean(0) for k in (0, 1)])
    a3 = _assign(C2)
    assert (a3 == a2).all()     # converged after 2 iterations
    return C0, a1, C1, a2, C2


def _walk_axes(dashed=True, letters=True):
    fig, ax = plt.subplots(figsize=(6, 6), dpi=220)
    _box(ax, ticks=True)
    if dashed:
        for x, y in _X:
            ax.plot([x, x], [0, y], ls=(0, (2.5, 3)), c=DASH, lw=0.9, zorder=1)
            ax.plot([0, x], [y, y], ls=(0, (2.5, 3)), c=DASH, lw=0.9, zorder=1)
    if letters:
        for n, (x, y) in POINTS.items():
            ax.text(x + 0.51, y + 0.47, n, ha="center", va="center",
                    fontsize=13, color=INK, zorder=4)
    return fig, ax


def _points(ax, assign=None):
    cols = GREY_STEPS if assign is None else [RED_P if a == 0 else GREEN_P for a in assign]
    ax.scatter(_X[:, 0], _X[:, 1], s=70, c=cols, edgecolor="none", zorder=3)


def _centroid(ax, xy, col, label=None, off=_INIT, alpha=1.0):
    ax.scatter(*xy, s=95, marker="P", c=col, edgecolor="white", lw=0.75,
               zorder=6, alpha=alpha)
    if label:
        ax.text(xy[0] + off[0], xy[1] + off[1], label, fontsize=16.5, color=col,
                ha="center", va="center", zorder=7)


def _move(ax, a, b, col):
    ax.annotate("", xy=b, xytext=a, zorder=5, arrowprops=dict(
        arrowstyle="-|>,head_length=0.35,head_width=0.14",
        color=col, lw=0.6, shrinkA=6, shrinkB=6, alpha=0.6))



C0, a1, C1, a2, C2 = kmeans_trace()


### Data points

In [ ]:
fig, ax = _walk_axes(dashed=False, letters=False); _points(ax)

glue("plot_kmeans_01_points", fig, display=False)


### Dataset with labels

In [ ]:
fig, ax = _walk_axes(); _points(ax)

glue("plot_kmeans_02_dataset", fig, display=False)


### Initialisation

In [ ]:
fig, ax = _walk_axes(); _points(ax)
_centroid(ax, C0[0], RED_C, _MU1); _centroid(ax, C0[1], GREEN_C, _MU2)

glue("plot_kmeans_03_init", fig, display=False)


### Iteration 1: E-step

In [ ]:
fig, ax = _walk_axes(); _points(ax, a1)
_centroid(ax, C0[0], RED_C, _MU1); _centroid(ax, C0[1], GREEN_C, _MU2)

glue("plot_kmeans_04_iter1_assign", fig, display=False)


### Iteration 1: M-step

In [ ]:
fig, ax = _walk_axes(); _points(ax, a1)
for k, col in ((0, RED_C), (1, GREEN_C)):
    _centroid(ax, C0[k], col, alpha=0.35); _move(ax, C0[k], C1[k], col)
_centroid(ax, C1[0], RED_C, _MU1, (-0.65, -0.65))
_centroid(ax, C1[1], GREEN_C, _MU2, (0.65, 0.65))

glue("plot_kmeans_05_iter1_update", fig, display=False)


### Iteration 2: E-step

In [ ]:
fig, ax = _walk_axes(); _points(ax, a2)
_centroid(ax, C1[0], RED_C, _MU1, _TL); _centroid(ax, C1[1], GREEN_C, _MU2, _TL)

glue("plot_kmeans_06_iter2_assign", fig, display=False)


### Iteration 2: M-step

In [ ]:
fig, ax = _walk_axes(); _points(ax, a2)
for k, col in ((0, RED_C), (1, GREEN_C)):
    _centroid(ax, C1[k], col, alpha=0.35); _move(ax, C1[k], C2[k], col)
_centroid(ax, C2[0], RED_C, _MU1, _TR); _centroid(ax, C2[1], GREEN_C, _MU2, _TR)

glue("plot_kmeans_07_iter2_update", fig, display=False)


### Converged

In [ ]:
fig, ax = _walk_axes(); _points(ax, a2)
_centroid(ax, C2[0], RED_C, _MU1, _TR); _centroid(ax, C2[1], GREEN_C, _MU2, _TR)

glue("plot_kmeans_08_converged", fig, display=False)
